## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, lower

## Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_payments")

## Silver Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Payment type normalization

In [0]:
df = df.withColumn("payment_type", lower(col("payment_type")))

### Remove invalid payment value

In [0]:
df = df.filter(col("payment_value") >= 0)

### Sanity checks of dataframe

In [0]:
df.limit(10).display()

## Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_payments")

### Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_payments LIMIT 10